In [1]:
import gc
# import torch

gc.collect()
# torch.cuda.empty_cache()
# torch.cuda.ipc_collect()

import os
os.listdir(); os.chdir("/aiau010_scratch/azm0269/clover/")

from clover.utils.utils import notebook_line_magic
notebook_line_magic()

In [2]:
import json
from pathlib import Path
import pandas as pd
import numpy as np


baselines = ["md3po", "ddpo", "b2diffurl", "dpok"]
eval_path = Path("outputs/")
training_data_eval_file = "training_metrics.json"
evaluation_data_eval_file = "eval_metrics.json"

def get_training_metrics_df(file_path):
    if file_path.exists():
        with open(file_path, "r") as f:
            list_of_metrics = json.load(f)
    else:
        print(f"File {file_path} does not exist. Skipping.")
        return pd.DataFrame()  # Return an empty DataFrame if the file doesn't exist
    all_metric_entries = []
    for entry in list_of_metrics:
        all_metric_entries.append(entry['metrics'])
    df = pd.DataFrame(all_metric_entries)
    return df

def get_eval_metrics_df(file_path):
    if file_path.exists():
        with open(file_path, "r") as f:
            list_of_metrics = json.load(f)
    else:
        print(f"File {file_path} does not exist. Skipping.")
        return pd.DataFrame()  # Return an empty DataFrame if the file doesn't exist
    
    scores = ['bert_reward', 'clip_reward']
    eval_summary = list()
    for entry in list_of_metrics:
        for score in scores:
            eval_summary.append({
                "value": round(np.array(entry[score]).mean(), 4),
                "score": score,
                "epoch": entry['epoch']
            })
    eval_df = pd.DataFrame(eval_summary)
    return eval_df
    
def get_metrics():
    training_eval_df = pd.DataFrame()
    eval_df = pd.DataFrame()
    for baseline in baselines:
        baseline_path = eval_path / baseline
        training_data_eval_file_path = baseline_path / "training_evals" / training_data_eval_file
        eval_data_file_path = baseline_path / "evals" / evaluation_data_eval_file

        
        df = get_training_metrics_df(training_data_eval_file_path)
        df["method"] = baseline
        training_eval_df = pd.concat([training_eval_df, df], ignore_index=True)

        df = get_eval_metrics_df(eval_data_file_path)
        df["method"] = baseline
        eval_df = pd.concat([eval_df, df], ignore_index=True)
    return training_eval_df, eval_df

In [17]:
training_eval_df, eval_df = get_metrics()

File outputs/b2diffurl/training_evals/training_metrics.json does not exist. Skipping.
File outputs/b2diffurl/evals/eval_metrics.json does not exist. Skipping.
File outputs/dpok/training_evals/training_metrics.json does not exist. Skipping.
File outputs/dpok/evals/eval_metrics.json does not exist. Skipping.


In [18]:
import matplotlib.pyplot as plt

eval_summary = eval_df.pivot_table(
    index=['method', 'score'],
    columns='epoch',
    values='value',
)
# eval_summary.T.iloc[:, 0].plot()
# plt.title("Bert Reward")
# plt.show()
# eval_summary.T.iloc[:, 1].plot()
# plt.title("Clip Reward")
# plt.show()
eval_summary.T.round(3)#.plot()

method        ddpo                   md3po            
score  bert_reward clip_reward bert_reward clip_reward
epoch                                                 
2.0          0.565       0.343       0.575       0.338
4.0          0.571       0.341       0.569       0.357
6.0          0.581       0.343       0.554       0.352
8.0          0.590       0.340       0.582       0.352
10.0         0.590       0.345       0.551       0.355
12.0         0.576       0.341       0.557       0.354
14.0         0.563       0.342       0.560       0.356
16.0         0.569       0.345       0.565       0.355
18.0         0.576       0.347       0.558       0.355
20.0         0.572       0.348       0.577       0.352
22.0         0.598       0.347       0.562       0.352
24.0         0.583       0.341       0.564       0.335
26.0         0.548       0.345       0.601       0.351
28.0         0.590       0.350       0.580       0.352
30.0         0.576       0.340       0.580       0.368
32.0         0.581       0.347       0.530       0.371
34.0         0.590       0.349       0.604       0.368
36.0         0.590       0.342       0.580       0.376
38.0         0.596       0.349       0.555       0.376
40.0         0.596       0.363       0.579       0.377
42.0         0.599       0.362         NaN         NaN
44.0         0.610       0.356         NaN         NaN
46.0         0.576       0.356         NaN         NaN
48.0         0.595       0.353         NaN         NaN
50.0         0.592       0.360         NaN         NaN

In [19]:
cols = ['advantage_mean', 'approx_kl', 'current_reward_mean', 'current_reward_std', 'epoch', 'loss', 'parameter_update_norm_mean',
    'raw_reward_mean', 'raw_reward_std', 'replay_reward_mean',
    'replay_reward_std', 'reward_mean', 'reward_std', 'selected_samples', 'skipped_updates','method']
training_eval_df[cols].pivot_table(
    index=['epoch'],
    columns=['method'],
    values=['reward_mean'], #'loss', 
    # aggfunc='mean'
)#.rolling(window=5).mean().plot(kind='line')

reward_mean          
method        ddpo     md3po
epoch                       
1.0       0.323986  0.323986
2.0       0.324598  0.324325
3.0       0.325123  0.325810
4.0       0.331960  0.327343
5.0       0.334801  0.333533
6.0       0.334177  0.331704
7.0       0.333805  0.335116
8.0       0.334508  0.334730
9.0       0.330327  0.335546
10.0      0.339265  0.342169
11.0      0.334645  0.334962
12.0      0.338115  0.339907
13.0      0.330987  0.335913
14.0      0.338720  0.336231
15.0      0.334859  0.343906
16.0      0.340074  0.343053
17.0      0.331185  0.338968
18.0      0.337821  0.341091
19.0      0.337039  0.341690
20.0      0.341753  0.341596
21.0      0.341400  0.345864
22.0      0.341496  0.344832
23.0      0.343946  0.347694
24.0      0.344436  0.345626
25.0      0.344075  0.346338
26.0      0.342336  0.341343
27.0      0.346192  0.347631
28.0      0.349647  0.346924
29.0      0.346570  0.349922
30.0      0.340755  0.345628
31.0      0.348372  0.348024
32.0      0.345251  0.340460
33.0      0.351218  0.342526
34.0      0.344419  0.336207
35.0      0.348634  0.335790
36.0      0.350061  0.336252
37.0      0.354270  0.338073
38.0      0.353311  0.335502
39.0      0.353471  0.338844
40.0      0.350196  0.340517
41.0      0.348876       NaN
42.0      0.356961       NaN
43.0      0.355868       NaN
44.0      0.354961       NaN
45.0      0.356893       NaN
46.0      0.353174       NaN
47.0      0.350327       NaN
48.0      0.354903       NaN
49.0      0.359746       NaN
50.0      0.354658       NaN

In [ ]:
import matplotlib.pyplot as plt

eval_summary = eval_df.pivot_table(
    index=['method', 'score'],
    columns='epoch',
    values='value',
)
eval_summary.T.iloc[:, 0].rolling(5).mean().plot()
plt.title("Bert Reward")
plt.show()
eval_summary.T.iloc[:, 1].rolling(5).mean().plot()
plt.title("Clip Reward")
plt.show()